# Dataset Analysis notebook


In [15]:
# load the csv and libraries
import pandas as pd
try:
    data = pd.read_csv('Exam_Score_Prediction.csv')
    print(data.head())
except FileNotFoundError:
    print("The file 'Exam_Score_Prediction.csv' was not found.")
    exit(1)


   student_id  age  gender   course  study_hours  class_attendance  \
0           1   17    male  diploma         2.78              92.9   
1           2   23   other      bca         3.37              64.8   
2           3   22    male     b.sc         7.88              76.8   
3           4   20   other  diploma         0.67              48.4   
4           5   20  female  diploma         0.89              71.6   

  internet_access  sleep_hours sleep_quality   study_method facility_rating  \
0             yes          7.4          poor       coaching             low   
1             yes          4.6       average  online videos          medium   
2             yes          8.5          poor       coaching            high   
3             yes          5.8       average  online videos             low   
4             yes          9.8          poor       coaching             low   

  exam_difficulty  exam_score  
0            hard        58.9  
1        moderate        54.8  
2       

In [16]:
# For each non-numeric column, print the unique values and their counts
for column in data.select_dtypes(include=['object']).columns:
    print(f"Column: {column}")
    print(data[column].value_counts())
    print()

Column: gender
gender
other     6726
male      6695
female    6579
Name: count, dtype: int64

Column: course
course
bca        2902
ba         2896
b.sc       2878
b.com      2864
bba        2836
diploma    2826
b.tech     2798
Name: count, dtype: int64

Column: internet_access
internet_access
yes    16988
no      3012
Name: count, dtype: int64

Column: sleep_quality
sleep_quality
average    6694
poor       6687
good       6619
Name: count, dtype: int64

Column: study_method
study_method
self-study       4079
online videos    4069
coaching         4036
group study      3922
mixed            3894
Name: count, dtype: int64

Column: facility_rating
facility_rating
medium    6760
low       6638
high      6602
Name: count, dtype: int64

Column: exam_difficulty
exam_difficulty
moderate    9878
easy        6141
hard        3981
Name: count, dtype: int64



note that since categories like course could be helpful, there are too many uniques so i will cut some categories based on amount of possible variables


In [17]:
dropped_courses = []
for column in data.select_dtypes(include=['object']).columns:
    unique_counts = data[column].nunique()
    print(f"Column: {column}, Unique Values: {unique_counts}")
    if unique_counts > 5:
        # If there are too many unique values, we can choose to drop them
        data = data.drop(columns=[column])
        dropped_courses.append(column)
print (f"Dropped columns: {dropped_courses}")

Column: gender, Unique Values: 3
Column: course, Unique Values: 7
Column: internet_access, Unique Values: 2
Column: sleep_quality, Unique Values: 3
Column: study_method, Unique Values: 5
Column: facility_rating, Unique Values: 3
Column: exam_difficulty, Unique Values: 3
Dropped columns: ['course']


For non numerical columns assign a numerical representation based on the unique values

In [18]:
info = {
    "category" : list(data.select_dtypes(include=['object']).columns),
    "unique_values" : {col: data[col].unique().tolist() for col in data.select_dtypes(include=['object']).columns}
}

# make into tree structure for easy reading

from rich.tree import Tree
from rich.console import Console

tree = Tree("Categories")
for category, values in info["unique_values"].items():
    branch = tree.add(category)
    for v in values:
        branch.add(str(v))

Console().print(tree)

Categories
├── gender
│   ├── male
│   ├── other
│   └── female
├── internet_access
│   ├── yes
│   └── no
├── sleep_quality
│   ├── poor
│   ├── average
│   └── good
├── study_method
│   ├── coaching
│   ├── online videos
│   ├── mixed
│   ├── self-study
│   └── group study
├── facility_rating
│   ├── low
│   ├── medium
│   └── high
└── exam_difficulty
    ├── hard
    ├── moderate
    └── easy

In [25]:
# Assign numerical representation to non-numerical columns
for row in data['gender'].index:
    if data.at[row, 'gender'] == 'male':
        data.at[row, 'gender'] = 0
    elif data.at[row, 'gender'] == 'female':
        data.at[row, 'gender'] = 1
    else:
        data.at[row, 'gender'] = 2
for row in data['internet_access'].index:
    if data.at[row, 'internet_access'] == 'no':
        data.at[row, 'internet_access'] = 0
    elif data.at[row, 'internet_access'] == 'yes':
        data.at[row, 'internet_access'] = 1
    else:
        data.at[row, 'internet_access'] = 2
for row in data['sleep_quality'].index:
    if data.at[row, 'sleep_quality'] == 'poor':
        data.at[row, 'sleep_quality'] = 0
    elif data.at[row, 'sleep_quality'] == 'average':
        data.at[row, 'sleep_quality'] = 1
    elif data.at[row, 'sleep_quality'] == 'good':
        data.at[row, 'sleep_quality'] = 2
    else:
        data.at[row, 'sleep_quality'] = 3
for row in data['study_method'].index:
    if data.at[row, 'study_method'] == 'coaching':
        data.at[row, 'study_method'] = 0
    elif data.at[row, 'study_method'] == 'online videos':
        data.at[row, 'study_method'] = 1
    elif data.at[row, 'study_method'] == 'mixed':
        data.at[row, 'study_method'] = 2
    elif data.at[row, 'study_method'] == 'self-study':
        data.at[row, 'study_method'] = 3
    elif data.at[row, 'study_method'] == 'group study':
        data.at[row, 'study_method'] = 4
for row in data['facility_rating'].index:
    if data.at[row, 'facility_rating'] == 'low':
        data.at[row, 'facility_rating'] = 0
    elif data.at[row, 'facility_rating'] == 'medium':
        data.at[row, 'facility_rating'] = 1
    elif data.at[row, 'facility_rating'] == 'high':
        data.at[row, 'facility_rating'] = 2
    else:
        data.at[row, 'facility_rating'] = 3
for row in data['exam_difficulty'].index:
    if data.at[row, 'exam_difficulty'] == 'easy':
        data.at[row, 'exam_difficulty'] = 0
    elif data.at[row, 'exam_difficulty'] == 'moderate':
        data.at[row, 'exam_difficulty'] = 1
    elif data.at[row, 'exam_difficulty'] == 'hard':
        data.at[row, 'exam_difficulty'] = 2
    else:
        data.at[row, 'exam_difficulty'] = 3

for column in data.select_dtypes(include=['object']).columns:
    data[column] = pd.to_numeric(data[column], errors='coerce')
data.head()

,student_id,age,gender,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,1,17,2,2.78,92.9,2,7.4,3,0,3,3,58.9
1,2,23,2,3.37,64.8,2,4.6,3,1,3,3,54.8
2,3,22,2,7.88,76.8,2,8.5,3,0,3,3,90.3
3,4,20,2,0.67,48.4,2,5.8,3,1,3,3,29.7
4,5,20,2,0.89,71.6,2,9.8,3,0,3,3,43.7


In [26]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        20000 non-null  int64  
 1   age               20000 non-null  int64  
 2   gender            20000 non-null  int64  
 3   study_hours       20000 non-null  float64
 4   class_attendance  20000 non-null  float64
 5   internet_access   20000 non-null  int64  
 6   sleep_hours       20000 non-null  float64
 7   sleep_quality     20000 non-null  int64  
 8   study_method      20000 non-null  int64  
 9   facility_rating   20000 non-null  int64  
 10  exam_difficulty   20000 non-null  int64  
 11  exam_score        20000 non-null  float64
dtypes: float64(4), int64(8)
memory usage: 1.8 MB


now to create training and testing split from data


In [27]:
# create the training and testing split
from sklearn.model_selection import train_test_split
X = data.drop('exam_score', axis=1)
y = data['exam_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
# save the training and testing splits to csv for later use
try:
    X_train.to_csv('X_train.csv', index=False)
    X_test.to_csv('X_test.csv', index=False)
    y_train.to_csv('y_train.csv', index=False)
    y_test.to_csv('y_test.csv', index=False)
except Exception as e:
    print(f"An error occurred while saving the files: {e}")